# 01. Carga y limpieza

Carga los JSON originales, elimina registros marcados como borrados, normaliza los campos comunes y guarda la base ejecutada en `data/processed/base.parquet`.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
PLOTS_DIR = PROJECT_ROOT / "outputs" / "plots"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

for directory in [DATA_PROCESSED_DIR, MODELS_DIR, PLOTS_DIR, METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


In [2]:
import pandas as pd
from IPython.display import display

from src.data_loader import load_json_files
from src.preprocessing import (
    clean_data,
    create_base_dataframe,
    extract_additional_fields,
    summarize_data_quality,
)


In [3]:
preventivo_raw, correctivo_raw = load_json_files(
    DATA_RAW_DIR / "preventivos.json",
    DATA_RAW_DIR / "correctivos.json",
)

clean_df = clean_data(preventivo_raw, correctivo_raw, empresa_id="VOY")
clean_df = extract_additional_fields(clean_df)
base_df = create_base_dataframe(clean_df, executed_only=True)
quality_df = summarize_data_quality(
    base_df,
    duplicate_subset=["firebase_id"],
)
base_path = DATA_PROCESSED_DIR / "base.parquet"
base_df.to_parquet(base_path, index=False)

print("Cantidad registros preventivo (raw):", len(preventivo_raw))
print("Cantidad registros correctivo (raw):", len(correctivo_raw))
print("Total registros ejecutados en base:", len(base_df))
print("Tipos de revisión ejecutados:")
print(base_df["tipo_revision"].value_counts(dropna=False))
print("Rango temporal:", base_df["fecha_evento"].min(), "→", base_df["fecha_evento"].max())
print("Buses únicos:", base_df["placa_patente"].nunique())
print("Columnas totales en base:", base_df.shape[1])
print("Parquet guardado en:", base_path)
display(quality_df)

preview_columns = [
    "firebase_id",
    "tipo_revision",
    "placa_patente",
    "fecha_evento",
    "pauta_proyectada",
    "causa_origen",
    "causa_origen_grouped",
    "sistema_componente",
    "taller_planta",
    "taller_planta_grouped",
    "repuestos_count",
    "repuestos_cantidad_total",
    "uuid_gestion_count",
]
preview_columns = [column for column in preview_columns if column in base_df.columns]
display(base_df[preview_columns].head())


Cantidad registros preventivo (raw): 636
Cantidad registros correctivo (raw): 14500
Total registros ejecutados en base: 15026
Tipos de revisión ejecutados:
tipo_revision
CORRECTIVO    14500
PREVENTIVO      526
Name: count, dtype: int64
Rango temporal: 2024-11-05 12:34:28 → 2025-12-31 20:53:00
Buses únicos: 735
Parquet guardado en: C:\Users\josep\Desktop\repo_mantenimiento_predictivo\data\processed\base.parquet


,firebase_id,tipo_revision,placa_patente,fecha_evento,pauta_proyectada,causa_origen,sistema_componente,taller_planta
0,nWclrrJce7aJJSN3QPUW,CORRECTIVO,GCBB90,2025-09-29 04:33:00,NaN,VARIOS,VARIOS,EL CONQUISTADOR
1,9790DYR77eVpWEnjY8UL,CORRECTIVO,GCBB90,2025-11-02 16:27:24,NaN,VARIOS,VARIOS,EL CONQUISTADOR
2,hrb4jliyx0wZ77NVu6z2,CORRECTIVO,GCBB90,2025-11-02 19:55:02,NaN,VARIOS,VARIOS,EL CONQUISTADOR
3,h2gedRg7wCDceaRvrS4k,CORRECTIVO,GCBB90,2025-11-11 21:19:15,NaN,VARIOS,VARIOS,EL CONQUISTADOR
4,dxlAmtlwe9gXeWENWddZ,CORRECTIVO,GCBB90,2025-12-23 21:01:00,NaN,VARIOS,VARIOS,EL CONQUISTADOR
